# RSA: dot product, ColBERT, cross-encoder and MUVERA

Choose **Runtime → Change runtime type → GPU**, then run all cells.

1. **Scoring quality:** normalized MiniLM dot product, ColBERT and a cross-encoder on exactly the same candidate rows. Predicate scores remain supervised task-specific controls.
2. **Approximation:** MUVERA Flat/HNSW versus exact full-corpus ColBERT, with separate relevance and top-K fidelity tables.
3. **Costs:** query encoding, pool search, scoring, total latency, corpus storage and build components.

The default labels are the existing CLIP image teacher, **not independent human judgments**. Cross-encoder scores are predictions, not truth. `judgment_pool.csv` exports the shared candidates for independent annotation.

Results go to a new v2 Drive directory. The completed v1 embeddings are imported automatically when available; their checksums/configuration are verified. Change the output directory after changing code/configuration. Logs retain complete tracebacks.


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

def run_logged(command, *, cwd=None, log_path):
    """Forward child stdout/stderr through notebook output and keep the failure tail."""
    import collections
    import subprocess
    import sys
    from pathlib import Path

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = collections.deque(maxlen=80)
    print(f'Python: {sys.version.split()[0]} | executable: {sys.executable}', flush=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- New invocation ---\n')
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f'Command exited with status {returncode}. Full log: {log_path}\n'
            + ''.join(tail))
    return returncode

drive.mount('/content/drive')
REPO = Path('/content/ras-late-interaction')
BRANCH = 'codex/colbert-muvera-baselines'
LOGS = Path('/content/drive/MyDrive/ras_late_interaction_logs')
if not REPO.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/hanialshater/ras.git', str(REPO)],
               log_path=LOGS / 'setup.log')
else:
    run_logged(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH],
               log_path=LOGS / 'setup.log')
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
run_logged([sys.executable, '-m', 'pip', 'install', '-e',
            str(REPO) + '[dev,benchmark,late-interaction]'],
           log_path=LOGS / 'install.log')


In [ ]:
import os
os.chdir(REPO)
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':' + str(REPO)
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_late_interaction.py', 'tests/test_retrieval_comparison.py'],
           cwd=REPO, log_path=LOGS / 'tests.log')


## Run the three experiments

This exploratory run uses 30 queries, a common pool of 5,000 products, one warmup and three measured requests per query/method. The cross-encoder processes each query/title pair again on each request, so this takes longer than the previous notebook.

Query encoding and cross-encoder inference use the available GPU. Dot products, reference MaxSim and HNSW use CPU. These are measured reference implementations, not optimized production serving. Latency is sequential (concurrency 1); 30 queries do not establish a reliable production p99.

Imported ColBERT build times belong to the original runtime. Set `PREPARED_FROM = None` to rebuild all embeddings on the current machine for a same-runtime build comparison.


In [ ]:
RUN = Path('/content/drive/MyDrive/ras_retrieval_comparison_seed7_v2')
PREPARED_FROM = Path('/content/drive/MyDrive/ras_late_interaction_seed7_v1')
# Set PREPARED_FROM = None for a fresh, same-runtime encoding/build comparison.
command = [sys.executable, '-u', '-m', 'experiments.retrieval_comparison',
           '--output-dir', str(RUN), '--queries', '30', '--seed', '7',
           '--fde-seed', '7', '--k', '50', '--pool-size', '5000',
           '--candidates', '100', '500', '1000', '2000', '5000',
           '--repetitions', '8', '--partition-bits', '4', '--fde-dim', '4096',
           '--backend', 'hnsw', '--warmup', '1', '--timing-repeats', '3',
           '--cross-encoder-checkpoint', 'cross-encoder/ms-marco-MiniLM-L6-v2',
           '--cross-encoder-batch-size', '32']
if PREPARED_FROM is not None and (PREPARED_FROM / 'prepared.complete.json').exists():
    command += ['--prepared-from', str(PREPARED_FROM)]
run_logged(command, cwd=REPO, log_path=LOGS / (RUN.name + '.log'))


## 1. Scoring quality on identical candidates

Recall here is conditional on the shared pool. Pool coverage reports how many full-corpus relevant products were available to any reranker. Paired deltas use query pairs, not timing repeats; a positive delta favors method A.


In [ ]:
import pandas as pd
print('Shared-pool scoring quality — CLIP teacher labels')
display(pd.read_csv(RUN / 'ranking_quality.csv'))
print('Shared-pool relevant-document coverage')
display(pd.read_csv(RUN / 'pool_coverage.csv').describe())
print('Paired nDCG differences with 95% bootstrap intervals')
deltas = pd.read_csv(RUN / 'paired_quality_deltas.csv')
display(deltas[deltas.metric == 'ndcg'])


## 2. MUVERA approximation versus exact ColBERT

ColBERT top-K recall measures reproduction of ColBERT's ordering, not semantic relevance. Full-corpus teacher relevance is displayed in a separate table. Candidate survival exposes post-filter losses. Operating points that miss a target are explicitly marked `not_reached`.


In [ ]:
print('Approximation fidelity')
display(pd.read_csv(RUN / 'approximation.csv'))
print('Full-corpus relevance — different recall denominator from shared pool')
display(pd.read_csv(RUN / 'retrieval_quality.csv'))
print('Exploratory latency at achieved mean ColBERT fidelity targets')
display(pd.read_csv(RUN / 'matched_fidelity.csv'))


## 3. Latency, storage and build time

`total_ms` includes query encoding, filtering, candidate retrieval and scoring. `scoring_ms` for the cross-encoder includes pair tokenization and transformer inference. Predicate timings use materialized scores and exclude actual predicate execution.

Storage tables count corpus artifacts, excluding model checkpoints and research replay files. HNSW serialization already contains an FDE copy. Model tensor payloads and whole-harness peak RSS are separate; this is not isolated per-method RAM. Inspect build provenance before comparing imported encoding times with current timings.


In [ ]:
import json
print('Latency: warmed sequential requests, concurrency=1')
latency = pd.read_csv(RUN / 'latency.csv')
display(latency[['scope', 'method', 'candidate_budget', 'samples',
                 'query_encoding_ms_p50', 'pool_search_ms_p50', 'scoring_ms_p50',
                 'total_ms_p50', 'total_ms_p95', 'total_ms_p99', 'timing_scope']])
print('Corpus storage, by pipeline')
display(pd.read_csv(RUN / 'storage.csv'))
print('Build components (seconds) and provenance')
display(pd.read_csv(RUN / 'build_time.csv'))
print('Corpus build totals (inspect provenance before comparing)')
display(pd.read_csv(RUN / 'build_totals.csv'))
print('Models/devices:', json.loads((RUN / 'models.json').read_text()))
print('Memory scope:', json.loads((RUN / 'memory.json').read_text()))


In [ ]:
from google.colab import files
files.download(str(RUN / 'late_interaction_results.zip'))
